In [1]:
import radiomics
from radiomics import featureextractor  # PyRadiomics extractor
import pandas as pd
import os
import glob
from tfrecordhandler import TFRecordDataHandler
import SimpleITK as sitk
import numpy as np
import SimpleITK as sitk


In [2]:
batch_size = 4

In [3]:
# Function to extract first-order features
def extract_first_order_features(image, mask):
    extractor = radiomics.firstorder.RadiomicsFirstOrder(image, mask)
    return extractor.execute()

# Function to extract 2D shape-based features
def extract_shape_2d_features(image, mask):
    # Ensure image and mask are 2D
    image_2d = sitk.GetImageFromArray(np.squeeze(sitk.GetArrayFromImage(image)))
    mask_2d = sitk.GetImageFromArray(np.squeeze(sitk.GetArrayFromImage(mask)))
    
    extractor = radiomics.shape2D.RadiomicsShape2D(image_2d, mask_2d, force2D=True)
    return extractor.execute()

# Functions to extract other 2D radiomic features
def extract_glcm_features(image, mask):
    extractor = radiomics.glcm.RadiomicsGLCM(image, mask)
    return extractor.execute()

def extract_glrlm_features(image, mask):
    extractor = radiomics.glrlm.RadiomicsGLRLM(image, mask)
    return extractor.execute()

def extract_glszm_features(image, mask):
    extractor = radiomics.glszm.RadiomicsGLSZM(image, mask)
    return extractor.execute()

def extract_ngtdm_features(image, mask):
    extractor = radiomics.ngtdm.RadiomicsNGTDM(image, mask)
    return extractor.execute()

def extract_gldm_features(image, mask):
    extractor = radiomics.gldm.RadiomicsGLDM(image, mask)
    return extractor.execute()

# Function to extract all radiomic features (without 3D features)
def extract_all_radiomic_features(image_np, mask_np):
    # Convert NumPy arrays to SimpleITK images
    image_sitk = sitk.GetImageFromArray(image_np)
    mask_sitk = sitk.GetImageFromArray(mask_np)

    # Extract and combine features from various categories
    feature_dicts = []
    feature_dicts.append(extract_first_order_features(image_sitk, mask_sitk))
    feature_dicts.append(extract_shape_2d_features(image_sitk, mask_sitk))
    feature_dicts.append(extract_glcm_features(image_sitk, mask_sitk))
    feature_dicts.append(extract_glrlm_features(image_sitk, mask_sitk))
    feature_dicts.append(extract_glszm_features(image_sitk, mask_sitk))
    feature_dicts.append(extract_ngtdm_features(image_sitk, mask_sitk))
    feature_dicts.append(extract_gldm_features(image_sitk, mask_sitk))
    
    # Combine all feature dictionaries into a single one
    combined_features = {}
    for feature_dict in feature_dicts:
        combined_features.update(feature_dict)
    
    return combined_features


# Initialize the TFRecord data handler
tfrecord = 'full_ds.tfrecord'
train_ds = TFRecordDataHandler(tfrecord, batch_size=batch_size, shuffle=True, augment=False)

# List to store the extracted features
all_features = []

# To store feature keys (header) for the CSV
feature_keys = None

# Iterate over the dataset, batch by batch
for images_tf, masks_tf, group_name, m_id, day_of_study in train_ds.dataset:
    # Convert each image and mask tensor in the batch to NumPy arrays
    for image_tensor, mask_tensor, group_name_tensor, m_id_tensor, day_of_study_tensor in zip(images_tf, masks_tf, group_name, m_id, day_of_study):
        image_np = image_tensor.numpy()  # Convert TensorFlow tensor to NumPy array
        mask_np = mask_tensor.numpy()    # Convert TensorFlow tensor to NumPy array

        # Handle potential errors during feature extraction
        try:
            # Extract radiomic features from the image and mask arrays
            features = extract_all_radiomic_features(image_np, mask_np)
            
            # Store feature keys (header) in the first iteration
            if feature_keys is None:
                feature_keys = list(features.keys())
                feature_keys.insert(0, 'group_name')
                feature_keys.insert(1, 'm_id')
                feature_keys.insert(2, 'day_of_study')
            
            # Append extracted features (values) to the list
            all_features.append(list(features.values()))
            all_features[-1].insert(0, group_name_tensor.numpy())
            all_features[-1].insert(1, m_id_tensor.numpy())
            all_features[-1].insert(2, day_of_study_tensor.numpy())

        except Exception as e:
            # Log error details for troubleshooting
            print(f"Error extracting features for image/mask. Image shape: {image_np.shape}, Mask shape: {mask_np.shape}. Error: {str(e)}")

# Convert the list of features to a pandas DataFrame
df_features = pd.DataFrame(all_features, columns=feature_keys)

# Save the features to a CSV file for machine learning
df_features.to_csv('radiomics_features.csv', index=False)


GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Avera

In [4]:
ds = TFRecordDataHandler(tfrecord, batch_size=batch_size, shuffle=True, augment=False)

print("Dataset size: ", ds.length )

Dataset size:  382
